## Scenario 3: Multiple data scientists working on multiple ML models

MLflow setup:
* Tracking server: yes, remote server (EC2).
* Backend store: postgresql database.
* Artifacts store: s3 bucket.

The experiments can be explored by accessing the remote server.

The exampe uses AWS to host a remote server. In order to run the example you'll need an AWS account. Follow the steps described in the file `mlflow_on_aws.md` to create a new AWS account and launch the tracking server. 

In [1]:
import mlflow
import os

from dotenv import load_dotenv
# Load the .env file
load_dotenv()

# Access AWS credentials from environment variables
aws_access_key = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
aws_region = os.getenv("AWS_DEFAULT_REGION")

print(f"AWS Region: {aws_region}")

TRACKING_SERVER_HOST = "ec2-13-38-118-156.eu-west-3.compute.amazonaws.com" # fill in with the public DNS of the EC2 instance
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:5000")

AWS Region: eu-west-3


In [2]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://ec2-13-38-118-156.eu-west-3.compute.amazonaws.com:5000'


In [3]:
mlflow.search_experiments() # list_experiments API has been removed, you can use search_experiments instead.()

[<Experiment: artifact_location='s3://mlflow-artifacts-distant/0', creation_time=1735834375297, experiment_id='0', last_update_time=1735834375297, lifecycle_stage='active', name='Default', tags={}>]

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2025/01/02 17:35:30 INFO mlflow.tracking.fluent: Experiment with name 'my-experiment-1' does not exist. Creating a new experiment.
2025/01/02 17:35:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


default artifacts URI: 's3://mlflow-artifacts-distant/1/e8054fc822354d73b0569a91893ce27c/artifacts'
🏃 View run overjoyed-squid-844 at: http://ec2-13-38-118-156.eu-west-3.compute.amazonaws.com:5000/#/experiments/1/runs/e8054fc822354d73b0569a91893ce27c
🧪 View experiment at: http://ec2-13-38-118-156.eu-west-3.compute.amazonaws.com:5000/#/experiments/1


In [5]:
mlflow.search_experiments()

[<Experiment: artifact_location='s3://mlflow-artifacts-distant/1', creation_time=1735835730085, experiment_id='1', last_update_time=1735835730085, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='s3://mlflow-artifacts-distant/0', creation_time=1735834375297, experiment_id='0', last_update_time=1735834375297, lifecycle_stage='active', name='Default', tags={}>]

### Interacting with the model registry

In [6]:
from mlflow.tracking import MlflowClient


client = MlflowClient(f"http://{TRACKING_SERVER_HOST}:5000")

In [9]:
client.search_registered_models()

[]

In [13]:
runs = client.search_runs(experiment_ids=['1'])
run_id = runs[0].info.run_id
print(f"run_id: {run_id}")

mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name='iris-classifier')

run_id: e8054fc822354d73b0569a91893ce27c


Successfully registered model 'iris-classifier'.
2025/01/02 17:41:48 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 1
Created version '1' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1735836108862, current_stage='None', description='', last_updated_timestamp=1735836108862, name='iris-classifier', run_id='e8054fc822354d73b0569a91893ce27c', run_link='', source='s3://mlflow-artifacts-distant/1/e8054fc822354d73b0569a91893ce27c/artifacts/models', status='READY', status_message='', tags={}, user_id='', version='1'>